<a href="https://colab.research.google.com/github/chitta-behera/Machine-Learning/blob/master/house_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch pandas scikit-learn

In [2]:
import torch
import torch.nn as nn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
data = {
    "Size":[1000,1500,1800,2400,3000,3500,4000,4500],
    "Bedrooms":[2,3,3,4,4,5,5,6],
    "Age":[20,15,10,8,5,3,2,1],
    "Price":[200,250,300,400,500,600,700,800]
}

df = pd.DataFrame(data)

In [4]:
df

,Size,Bedrooms,Age,Price
0,1000,2,20,200
1,1500,3,15,250
2,1800,3,10,300
3,2400,4,8,400
4,3000,4,5,500
5,3500,5,3,600
6,4000,5,2,700
7,4500,6,1,800


In [5]:
X = df[["Size","Bedrooms","Age"]].values
y = df["Price"].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
print(type(X_train))

<class 'numpy.ndarray'>


In [9]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train).view(-1,1)
y_test = torch.FloatTensor(y_test).view(-1,1)

In [10]:
X_train

tensor([[-1.4752, -1.5492,  1.9447],
        [ 1.4200,  1.5492, -1.0512],
        [-0.8134, -0.7746,  0.3679],
        [ 0.1792,  0.0000, -0.4205],
        [-0.3171,  0.0000,  0.0526],
        [ 1.0064,  0.7746, -0.8935]])

In [11]:
class HousePriceModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(3,16),
            nn.ReLU(),

            nn.Linear(16,8),
            nn.ReLU(),

            nn.Linear(8,1)
        )

    def forward(self,x):
        return self.network(x)

In [12]:
model = HousePriceModel()

In [13]:
criterion = nn.MSELoss()

In [14]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [15]:
epochs = 500

for epoch in range(epochs):

    prediction = model(X_train)

    loss = criterion(prediction, y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch} Loss {loss.item():.2f}"
        )

Epoch 0 Loss 278414.94
Epoch 50 Loss 268492.47
Epoch 100 Loss 164639.02
Epoch 150 Loss 18107.82
Epoch 200 Loss 4176.47
Epoch 250 Loss 2019.59
Epoch 300 Loss 916.63
Epoch 350 Loss 387.49
Epoch 400 Loss 164.15
Epoch 450 Loss 79.49


In [17]:
with torch.no_grad():

    prediction = model(X_test)

    print("Actual Price")
    print(y_test)

    print("Predicted Price")
    print(prediction)

Actual Price
tensor([[250.],
        [600.]])
Predicted Price
tensor([[231.4787],
        [655.0390]])


In [19]:
new_house = [[2800,4,6]]

new_house = scaler.transform(new_house)


new_house = torch.FloatTensor(new_house)

with torch.no_grad():

    price = model(new_house)

print(price.item())

467.97491455078125
